# NVD ↔ EPSS Parser

Phase 0 prerequisite-audit deliverable (see *Roadmap - Cybersecurity_VulDet_Agetntic AI.md*, Phase 0 table): **"Write a script that parses NVD JSON feeds and joins against EPSS CSV."**

Pipeline:
1. Pull CVE records from the **NVD CVE API 2.0** (`services.nvd.nist.gov`) — NVD's own data-feeds page states the API is now the preferred way to stay current; the static year-based JSON feed files are the legacy fallback.
2. Pull the matching daily **EPSS** scores CSV from the [`empiricalsec/epss_scores`](https://github.com/empiricalsec/epss_scores) mirror (a plain `cve,epss,percentile` file, gzip-compressed per day).
3. Join the two on CVE ID and rank by exploitation probability.

Sources referenced: https://nvd.nist.gov/vuln/data-feeds#APIS and https://github.com/empiricalsec/epss_scores/tree/main

In [1]:
import datetime as dt
import gzip
import io
import os
import time

import pandas as pd
import requests

NVD_API_BASE = "https://services.nvd.nist.gov/rest/json/cves/2.0"
EPSS_RAW_BASE = "https://raw.githubusercontent.com/empiricalsec/epss_scores/main"

# Optional: set an NVD API key (https://nvd.nist.gov/developers/request-an-api-key)
# to raise the rate limit from 5 to 50 requests / 30s.
NVD_API_KEY = os.environ.get("NVD_API_KEY")

## 1. Fetch CVEs from the NVD API

`resultsPerPage` caps at 2000, so multi-page result sets are paginated via `startIndex`. The API enforces a rolling rate limit, so we sleep between pages (0.6s with an API key, 6s without).

In [ ]:
# fetch_nvd_cves
# ----------------
# Talks to the NVD CVE API 2.0 and pulls every CVE published within
# [pub_start_date, pub_end_date].
#
# How it works:
#   - Builds the query params (pubStartDate/pubEndDate/resultsPerPage/startIndex)
#     and, if an API key is supplied, adds it to the request headers.
#   - The API returns results a page at a time (capped at `results_per_page`,
#     max 2000) along with a `totalResults` count, so the function loops,
#     bumping `start_index` by `results_per_page` each iteration, until
#     `start_index` has caught up with `totalResults`.
#   - Each page's `vulnerabilities` list wraps the actual CVE object under a
#     "cve" key — those inner dicts are what get collected into `all_cves`.
#   - NVD rate-limits requests (5/30s unauthenticated, 50/30s with an API
#     key), so the loop sleeps between pages: 0.6s with a key, 6s without.
#
# Params:
#   pub_start_date / pub_end_date - datetime window (inclusive/exclusive per
#       the API) to filter CVEs by publication date.
#   api_key - optional NVD API key; raises the rate limit and shortens sleeps.
#   results_per_page - page size requested from the API (max 2000).
#
# Returns: a flat list of raw CVE JSON objects (dicts), unparsed.
def fetch_nvd_cves(pub_start_date: dt.datetime, pub_end_date: dt.datetime,
                    api_key: str | None = None, results_per_page: int = 2000) -> list[dict]:
    """Fetch raw CVE objects from the NVD API 2.0 for a publication date window."""
    headers = {"apiKey": api_key} if api_key else {}
    sleep_s = 0.6 if api_key else 6.0

    all_cves = []
    start_index = 0
    total_results = None

    while total_results is None or start_index < total_results:
        params = {
            "pubStartDate": pub_start_date.strftime("%Y-%m-%dT%H:%M:%S.000"),
            "pubEndDate": pub_end_date.strftime("%Y-%m-%dT%H:%M:%S.000"),
            "resultsPerPage": results_per_page,
            "startIndex": start_index,
        }
        resp = requests.get(NVD_API_BASE, params=params, headers=headers, timeout=30)
        resp.raise_for_status()
        payload = resp.json()

        total_results = payload["totalResults"]
        all_cves.extend(v["cve"] for v in payload.get("vulnerabilities", []))

        start_index += results_per_page
        if start_index < total_results:
            time.sleep(sleep_s)

    return all_cves

In [ ]:
# _extract_cvss
# --------------
# Helper that digs the "best" CVSS score out of a single CVE's raw `metrics`
# dict (the `cve["metrics"]` blob from the NVD API response).
#
# How it works:
#   - NVD may publish multiple CVSS versions for the same CVE (v2, v3.0,
#     v3.1) depending on when it was scored. This walks the preference order
#     v3.1 -> v3.0 -> v2 and returns as soon as it finds a non-empty list for
#     one of those keys, since v3.1 is the most current/detailed scheme.
#   - Each metrics entry is a list (NVD sometimes reports multiple sources
#     per version), so only the first entry (`entries[0]`) is used.
#   - `baseSeverity` is preferred from the entry itself, falling back to the
#     nested `cvssData` if not present there (API shape differs slightly
#     between CVSS versions).
#
# Params: metrics - the `cve["metrics"]` dict from a raw NVD CVE object.
#
# Returns: a dict with keys cvss_version, base_score, base_severity,
#   vector_string. If none of the three CVSS keys are populated (metrics
#   missing entirely), all four values are None.
def _extract_cvss(metrics: dict) -> dict:
    """Prefer CVSS v3.1, fall back to v3.0, then v2 — whichever NVD populated first."""
    for key in ("cvssMetricV31", "cvssMetricV30", "cvssMetricV2"):
        entries = metrics.get(key)
        if entries:
            data = entries[0]["cvssData"]
            return {
                "cvss_version": data.get("version"),
                "base_score": data.get("baseScore"),
                "base_severity": entries[0].get("baseSeverity", data.get("baseSeverity")),
                "vector_string": data.get("vectorString"),
            }
    return {"cvss_version": None, "base_score": None, "base_severity": None, "vector_string": None}


# parse_nvd_cves
# --------------
# Converts the list of raw, deeply-nested CVE JSON objects returned by
# fetch_nvd_cves() into a flat, tabular pandas DataFrame suitable for
# joining against the EPSS scores.
#
# How it works:
#   - For each raw CVE dict, pulls out the English-language description from
#     the `descriptions` list (NVD stores descriptions in multiple
#     languages; `next(...)` grabs the first one tagged "en", defaulting to
#     an empty string if none exists).
#   - Grabs the top-level `id`, `published`, and `lastModified` fields
#     as-is (ISO timestamp strings).
#   - Delegates to `_extract_cvss` to flatten whichever CVSS metric version
#     is available into the same row via dict unpacking (`**`).
#   - Collects one dict per CVE into `rows`, then hands the whole list to
#     `pd.DataFrame` to build the table in one shot.
#
# Params: raw_cves - list of raw CVE dicts, as returned by fetch_nvd_cves().
#
# Returns: a DataFrame with one row per CVE and columns cve_id, published,
#   last_modified, description, cvss_version, base_score, base_severity,
#   vector_string.
def parse_nvd_cves(raw_cves: list[dict]) -> pd.DataFrame:
    """Flatten raw NVD CVE JSON objects into a tabular DataFrame."""
    rows = []
    for cve in raw_cves:
        en_desc = next((d["value"] for d in cve.get("descriptions", []) if d["lang"] == "en"), "")
        rows.append({
            "cve_id": cve["id"],
            "published": cve.get("published"),
            "last_modified": cve.get("lastModified"),
            "description": en_desc,
            **_extract_cvss(cve.get("metrics", {})),
        })
    return pd.DataFrame(rows)

## 2. Fetch the matching EPSS scores CSV

Files are published daily as `epss_scores-YYYY-MM-DD.csv.gz` under a per-year folder. If a given date isn't there yet (publishing lag), step back a day at a time.

In [ ]:
# fetch_epss_scores
# ------------------
# Downloads the daily EPSS (Exploit Prediction Scoring System) scores CSV
# for a given date, walking backwards day-by-day if that exact date isn't
# published yet.
#
# How it works:
#   - EPSS files are published once per day at a predictable URL pattern:
#     `{EPSS_RAW_BASE}/{year}/epss_scores-{YYYY-MM-DD}.csv.gz`.
#   - Publishing can lag, so the function starts at `date` and steps
#     backwards one day at a time (via `offset`), trying each URL in turn,
#     up to `max_lookback_days` attempts.
#   - The first URL that returns HTTP 200 is treated as found: its gzip
#     content is decompressed in-memory (`io.BytesIO` + `gzip.open`) and
#     parsed as CSV directly into a DataFrame, skipping any lines that
#     start with a comment marker (`comment="#"`).
#   - The actual date used (which may be earlier than the requested `date`
#     if there was a lookback) is stamped onto `df.attrs["score_date"]` so
#     callers know exactly which day's scores they got.
#   - If no file is found within `max_lookback_days` attempts, raises
#     FileNotFoundError rather than silently returning stale/empty data.
#
# Params:
#   date - the preferred (usually most recent) date to fetch scores for.
#   max_lookback_days - how many days to step backwards before giving up.
#
# Returns: a DataFrame of the CSV contents (columns cve, epss, percentile),
#   with `df.attrs["score_date"]` set to the ISO date actually used.
def fetch_epss_scores(date: dt.date, max_lookback_days: int = 5) -> pd.DataFrame:
    """Download and decompress the daily EPSS scores CSV for the nearest available date <= `date`."""
    for offset in range(max_lookback_days):
        d = date - dt.timedelta(days=offset)
        url = f"{EPSS_RAW_BASE}/{d.year}/epss_scores-{d.isoformat()}.csv.gz"
        resp = requests.get(url, timeout=30)
        if resp.status_code == 200:
            with gzip.open(io.BytesIO(resp.content), "rt") as f:
                df = pd.read_csv(f, comment="#")
            df.attrs["score_date"] = d.isoformat()
            return df
    raise FileNotFoundError(
        f"No EPSS scores file found in the {max_lookback_days} days up to {date.isoformat()}"
    )

## 3. Run the pipeline

By default this pulls CVEs *published* in the last 7 days (a full historical backfill is the same call with an earlier `pub_start_date`, paginated in ≤90-day windows since the API caps date ranges at 120 days) and joins them against the most recent EPSS score file.

In [5]:
today = dt.date.today()

raw_cves = fetch_nvd_cves(
    pub_start_date=dt.datetime.combine(today - dt.timedelta(days=7), dt.time.min),
    pub_end_date=dt.datetime.combine(today, dt.time.min),
    api_key=NVD_API_KEY,
)
nvd_df = parse_nvd_cves(raw_cves)

epss_df = fetch_epss_scores(today - dt.timedelta(days=1))

print(f"NVD CVEs fetched: {len(nvd_df)}")
print(f"EPSS score date used: {epss_df.attrs['score_date']} ({len(epss_df)} rows)")

NVD CVEs fetched: 1780
EPSS score date used: 2026-07-28 (353212 rows)


In [6]:
joined = nvd_df.merge(epss_df, left_on="cve_id", right_on="cve", how="left").drop(columns="cve")

matched = joined["epss"].notna().sum()
print(f"Matched {matched}/{len(joined)} NVD CVEs to an EPSS score "
      f"({matched / len(joined):.1%})")

joined.sort_values("epss", ascending=False).head(10)[
    ["cve_id", "base_score", "base_severity", "epss", "percentile"]
]

Matched 1558/1780 NVD CVEs to an EPSS score (87.5%)


,cve_id,base_score,base_severity,epss,percentile
94,CVE-2026-62144,9.1,CRITICAL,0.20623,0.97262
68,CVE-2026-16232,9.1,CRITICAL,0.12685,0.95846
95,CVE-2026-62145,7.5,HIGH,0.07540,0.93871
515,CVE-2026-6516,10.0,CRITICAL,0.04729,0.90914
501,CVE-2026-47668,10.0,CRITICAL,0.04339,0.90205
1241,CVE-2026-17191,9.1,CRITICAL,0.02826,0.85137
739,CVE-2026-65711,7.2,HIGH,0.02406,0.82358
1242,CVE-2026-17192,8.5,HIGH,0.02338,0.81846
1120,CVE-2026-45112,7.5,HIGH,0.01940,0.78034
524,CVE-2026-47670,NaN,None,0.01714,0.75048


## 4. Save the joined table

In [7]:
output_path = f"nvd_epss_joined_{today.isoformat()}.csv"
joined.sort_values("epss", ascending=False).to_csv(output_path, index=False)
print(f"Wrote {len(joined)} rows to {output_path}")

Wrote 1780 rows to nvd_epss_joined_2026-07-29.csv
